# Pendências

Notebook destinado a investigar algumas pendências que ficaram depois da ingestão dos dados. Sendo elas:

- como aparece o `codigo_escola` mascarado
- `item_adaptado` em 2021
- não resposta no questionário

In [ ]:
from pathlib import Path
import duckdb

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PQ = ROOT / "data" / "parquet"

base = f"{PQ}/base_unica/**/*.parquet"
res = f"{PQ}/resultados/**/*.parquet"
part = f"{PQ}/participantes/**/*.parquet"
itens = f"{PQ}/itens/**/*.parquet"

con = duckdb.connect()
print(ROOT)

### Pendência 1: como o `codigo_escola` aparece mascarado

**Objetivo**: Descobrir como o INEP escondeu o código das escolas pequenas nos dados de 2024 e 2025.

**Contexto**: A partir de 2024, a tabela RESULTADOS traz o codigo_escola, que identifica a escola de cada participante. Para proteger a privacidade dos alunos, o INEP mascara esse código quando a escola tem menos de 10 participantes. Numa escola com 3 alunos fazendo o ENEM, saber a escola e a nota seria quase o mesmo que saber a nota de cada aluno.

**O problema**: No `ingestao_nulos.csv`, o `codigo_escola` tem **exatamente** o mesmo percentual de nulos que o município da escola:

| Coluna                      | 2024           | 2025       |
|-----------------------------|----------------|------------|
|`nome_municipio_escola`| 66,98% | 62,93%|
|`codigo_escola` | 66,98% | 62,93%|

Se o mascaramento deixasse o código vazio, o `codigo_escola` teria mais nulos que o município, porque as escolas pequenas teriam o município preenchido e o código vazio. Como os percentuais são idênticos, as escolas mascaradas não estão como nulo: o INEP colocou algum outro valor no lugar do código.

Identificar esses valores falsos é importante porque, se não forem tratados adequadamente, ele vira uma "escola" falsa na análise.


In [ ]:
con.sql(f"""
    SELECT ano, codigo_escola, COUNT(*) AS n
    FROM '{res}'
    WHERE codigo_escola IS NOT NULL
    GROUP BY ALL
    ORDER BY n DESC
    LIMIT 10
""").show()

A tabela mostra que não existe um valor substituto para o código das escolas pequenas. Como pode ser visto acima, os códigos mais frequentes são de escolas reais:
    - todos tem 8 dígitos e começam com 31, que é o código do IBGE para Minas Gerais;
    - O maior tem 554 participantes, um número plausível para uma escola estadual grande;
    - Os mesmos códigos aparecem em 2024 e 2025 com contagens parecidas, como `31245488` e `31167436`.

Se existisse um código de mascaramento, ele estaria no topo da tabela (considerando que ele fosse igualmente aplicado para todas as escolas com menos de 10 participantes).

Agora levanto outra **hipótese**: as escolas mascaradas provavelmente aparecem como nulo também no município e nas outras colunas de escola, e não só no código. Assim os percentuais idênticos mostrados na ingestao_nulos.csv também fariam sentido.

In [ ]:
con.sql(f"""
    SELECT ano,
        COUNT(*) FILTER (WHERE codigo_escola IS NULL AND nome_municipio_escola IS NOT NULL) AS codigo_nulo_municipio_ok,
        COUNT(*) FILTER (WHERE codigo_escola IS NOT NULL AND nome_municipio_escola IS NULL) AS codigo_ok_municipio_nulo
    FROM '{res}' GROUP BY ano ORDER BY ano
""").show()

Assim confirmamos que o código e o município da escola estão sempre preenchidos juntos ou vazio juntos em 2024 e 2025. Para confirmar, vamos fazer mais duas análises. Primeiro, vamos investigar se existem escolas com menos de 10 participantes.

In [ ]:
con.sql(f"""
    WITH por_escola AS (
        SELECT ano, codigo_escola, COUNT(*) AS n
        FROM '{res}' WHERE codigo_escola IS NOT NULL
        GROUP BY ALL
    )
    SELECT ano, MIN(n) AS menor_escola,
        COUNT(*) FILTER (WHERE n < 10) AS escolas_com_menos_de_10,
        COUNT(*) AS total_escolas
    FROM por_escola GROUP BY ano ORDER BY ano
""").show()

O resultado confirma que existem sim escolas com menos de 10 participantes nas provas do ENEM de 2024 e 2025. Chegando ao limite de escolas com apenas 1 participante por edição da prova. Se o INEP mascarasse toda escola com menos de 10 participantes, essas escolas não deveriam aparecer no código.

Depois de um longo período de considerações, reformulei a hipótese. Como a D003 estipulou que a tabela RESULTADOS fosse filtrada por prova **ou** escola em MG, ela também contém alunos de escolas de outros estados que fizeram a prova em Minas, e essas escolas provavelmente devem aparecer com poucos participantes, porque só vemos os alunos delas que vieram fazer a prova aqui. Já para as escolas mineiras, a contagem é completa, já que todos os alunos delas entram pelo filtro da escola.

Vamos então, realizar uma nova consulta agora separando as escolas mineiras das escolas de outros estados.

In [ ]:
con.sql(f"""
    WITH por_escola AS (
        SELECT ano, codigo_escola,
            sigla_estado_escola = 'MG' AS escola_mg,
            COUNT(*) AS n
        FROM '{res}' WHERE codigo_escola IS NOT NULL
        GROUP BY ALL
    )
    SELECT ano, escola_mg,
        MIN(n) AS menor_escola,
        COUNT(*) FILTER (WHERE n < 10) AS escolas_com_menos_de_10,
        COUNT(*) AS total_escolas
    FROM por_escola
    GROUP BY ALL ORDER BY ano, escola_mg
""").show()

Uma parte da última hipótese foi sanada. Para as escolas de fora do estado (`escola_mg = false`), conseguimos ver que mais de 90% delas apresenta menos de 10 alunos fazendo o ENEM no território de Minas Gerais. 

| Ano | Escolas de outros estados | Com menos de 10 |
|------|--------------------------|-----------------|
|2024 | 263 | 242 (92%)|
|2025 | 274 | 251 (92%) |

Isso gera uma nova decisão para a próxima estapa deste projeto: **as análises por escola devem usar apenas as escolas mineiras (`sigla_estado_escola = 'MG'`)**, porque as de fora aparecem incompletas.

Por outro lado, a tabela acima ainda mostra que existem escolas mineiras pequenas que não foram mascaradas.

| Ano | Escolas de MG | Com menos de 10 |
|------|--------------------------|-----------------|
|2024 | 3.326 | 565 (17%)|
|2025 | 3.497 | 489 (14%) |

Então a regra registrada na D002 ("mascarado se a escola tem menos de 10 participantes") não aparece nos dados da forma como foi escrita.

Resolvi reabrir o dicionário de dados de 2024 para verificar se havia alguma observação que eu deixei passar e sim, havia uma observação. O texto que informava sobre a máscara no dicionário diz exatamente o seguinte:

> Informação obtida da base de matrículas do Censo Escolar 2024, a partir do CPF do participante e filtrando as etapas de ensino de prováveis concluintes do ensino médio (27,28,29,37,38,71,32,33,34). O código real será substituído por uma máscara, caso escola tenha menos de 10 participantes na edição do exame.

Como já investigamos a possibilidade uma máscara única ser usada para as escolas com menos de 10 participantes e não encontramos tal máscara, o texto do INEP sugere que as máscaras utilizadas são únicas e que continuam parecendo códigos de escolas reais. Por isso a busca pelo valor mais frequente feita no início dessa investigação não achou nada. Vamos verificar se, de fato, as máscaras seguem o mesmo padrão dos códigos reais das escolas com mais participantes.

In [ ]:
con.sql(f"""
    WITH por_escola AS (
        SELECT ano, codigo_escola, COUNT(*) AS n
        FROM '{res}'
        WHERE codigo_escola IS NOT NULL AND sigla_estado_escola = 'MG'
        GROUP BY ALL
    )
    SELECT ano,
        n < 10 AS menos_de_10,
        length(codigo_escola) AS tamanho,
        codigo_escola LIKE '31%' AS comeca_com_31,
        COUNT(*) AS escolas,
        MIN(codigo_escola) AS exemplo_min,
        MAX(codigo_escola) AS exemplo_max
    FROM por_escola
    GROUP BY ALL ORDER BY ano, menos_de_10, tamanho
""").show()

Essa última análise resolve a pendência 1. O resultado mostra que as escolas com menos de 10 participantes têm o código substituído por uma máscara no formato `600xxxxx`. Os códigos reais, como falado anteriormente, começam com o código da UF no IBGE (31 para Minas Gerais).

**Conclusões**
- A máscara é identificável pelo formato, sem precisar contar o número de participantes.
- Cada escola mascarada recebe uma máscara própria.
- As demais colunas da escola são reais: município, rede e localização continuam valendo para esses alunos.